To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News


Unsloth's [Docker image](https://hub.docker.com/r/unsloth/unsloth) is here! Start training with no setup & environment issues. [Read our Guide](https://docs.unsloth.ai/new/how-to-train-llms-with-unsloth-and-docker).

[gpt-oss RL](https://docs.unsloth.ai/new/gpt-oss-reinforcement-learning) is now supported with the fastest inference & lowest VRAM. Try our [new notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/gpt-oss-(20B)-GRPO.ipynb) which creates kernels!

Introducing [Vision](https://docs.unsloth.ai/new/vision-reinforcement-learning-vlm-rl) and [Standby](https://docs.unsloth.ai/basics/memory-efficient-rl) for RL! Train Qwen, Gemma etc. VLMs with GSPO - even faster with less VRAM.

Unsloth now supports Text-to-Speech (TTS) models. Read our [guide here](https://docs.unsloth.ai/basics/text-to-speech-tts-fine-tuning).

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.55.4
!pip install --no-deps trl==0.22.2

### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 8146 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth_zoo/__init__.py:514: UserWarning: Unsloth fused-forward install skipped: requires transformers >= 4.56.0.
  _install_fused_forward()


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.3: Fast Llama patching. Transformers: 4.55.4.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
!pip install --upgrade torchao
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Already have LoRA adapters! We shall skip this step.


<a name="Data"></a>
### Data Prep
We now use the Alpaca dataset from [yahma](https://huggingface.co/datasets/yahma/alpaca-cleaned), which is a filtered version of 52K of the original [Alpaca dataset](https://crfm.stanford.edu/2023/03/13/alpaca.html). You can replace this code section with your own data prep.

**[NOTE]** To train only on completions (ignoring the user's input) read TRL's docs [here](https://huggingface.co/docs/trl/sft_trainer#train-on-completions-only).

**[NOTE]** Remember to add the **EOS_TOKEN** to the tokenized output!! Otherwise you'll get infinite generations!

If you want to use the `llama-3` template for ShareGPT datasets, try our conversational [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Alpaca.ipynb)

For text completions like novel writing, try this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Mistral_(7B)-Text_Completion.ipynb).

In [ ]:
#abrir ventana para cargar dataset
from google.colab import files
uploaded = files.upload()

Saving legal_dataset_final.json to legal_dataset_final.json


In [ ]:
#cargar dataset
import json

with open("legal_dataset_final.json", "r", encoding="utf-8-sig") as f:
    data = json.load(f)

print(f"Se cargaron {len(data)} ejemplos.")

Se cargaron 453 ejemplos.


In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

train_data, val_data = train_test_split(
    data,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"Entrenamiento: {len(train_dataset)}")
print(f"Validación: {len(val_dataset)}")

Entrenamiento: 362
Validación: 91


In [ ]:
alpaca_prompt = """Redacta un informe legal siguiendo estrictamente la estructura:

I. ANTECEDENTES:
II. BASE LEGAL:
III. ANÁLISIS:
IV. CONCLUSIÓN:
V. RECOMENDACIONES:

Asegúrate de *no terminar el texto sin incluir todas las secciones*.
Finaliza siempre con RECOMENDACIONES.

### Instruction:
{}

### Input:
{}

### Output:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    texts = []
    # Check if 'examples' is a single dictionary (non-batched) or a dictionary of lists (batched)
    if isinstance(examples['instruction'], str): # Single example case
        instruction = examples['instruction']
        input_text  = examples['input']
        output_text = examples['output']
        text = alpaca_prompt.format(instruction, input_text, output_text) + EOS_TOKEN
        texts.append(text)
    else: # Batched case
        instructions = examples["instruction"]
        inputs       = examples["input"]
        outputs      = examples["output"]
        for instruction, input, output in zip(instructions, inputs, outputs):
            text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
            texts.append(text)
    return texts

#importar dataset
from datasets import Dataset
from datasets import load_dataset
# dataset = Dataset.from_list(train_dataset)
# dataset = dataset.map(formatting_prompts_func, batched = True,)

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import EarlyStoppingCallback
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field = "text",
    max_seq_length = 1024,
    packing = False, # Can make training 5x faster for short sequences.
    formatting_func = formatting_prompts_func,
    # Temporarily removed EarlyStoppingCallback due to incompatibility with SFTConfig
    # callbacks=[
    #     EarlyStoppingCallback(early_stopping_patience=3)
    # ],

    args = SFTConfig(
        per_device_train_batch_size = 2, # Reduced from 2 to 1 to save memory
        gradient_accumulation_steps = 4, # Increased from 4 to 8 to compensate for smaller batch size
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 25,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
        output_dir = "outputs",
        report_to = "none" # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/362 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/91 [00:00<?, ? examples/s]

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-80GB. Max memory = 79.318 GB.
7.135 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 362 | Num Epochs = 1 | Total steps = 25
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
0,0.805500,0.765408


In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Aprobacion de plan de gobierno digital", # instruction
        "INFORME N°094-2025-UEI/OPP/MPC-BPR \nA        : Lic. Edder Joshimar Warthon Gamarra \nJefe de la Oficina de Planificación y Presupuesto \nDE      : Brady Palma Rodríguez \nEncargado de la Unidad de Estadística e Informática \nASUNTO : Solicitud de informe legal para aprobación del Plan de Gobierno Digital \nREFERENCIA : Plan de Gobierno Digital – Municipalidad Provincial de Calca, versión \npreliminar 2025 \nFECHA : Calca, 02 de agosto del 2025 \nMe dirijo a usted para saludarlo cordialmente y, a la vez, informar que la Unidad de \nEstadística e Informática ha culminado la elaboración del documento preliminar del Plan \nde Gobierno Digital de la Municipalidad Provincial de Calca, correspondiente al \nperiodo 2025–2027, en el marco del cumplimiento de la Política Nacional de \nTransformación Digital y las disposiciones emitidas por la Secretaría de Gobierno y \nTransformación Digital de la PCM. \nANTECEDENTES \nLa implementación del Plan de Gobierno Digital es una acción estratégica que responde a \nlos lineamientos establecidos por el Decreto Supremo N.º 029-2021-PCM y la Ley N.º \n1412, Ley de Gobierno Digital. \nEn este sentido, se ha trabajado un documento técnico que contiene los objetivos, líneas \nde acción, metas, responsables y cronograma de actividades, que busca fortalecer el \necosistema digital institucional, garantizar la interoperabilidad, impulsar la identidad y \nservicios digitales, y promover la seguridad digital. \nCabe señalar que la aprobación del presente plan requiere contar previamente con el \ninforme legal correspondiente, el cual determine la viabilidad normativa y sujeción a las \ndisposiciones vigentes. \nANÁLISIS \nLa emisión del informe legal permitirá sustentar formal y legalmente la aprobación del \nPlan de Gobierno Digital, a fin de remitirlo posteriormente al Pleno del Concejo Municipal, \ny así cumplir con su implementación de acuerdo con la normativa nacional. \nEste procedimiento es imprescindible para garantizar la legalidad de las acciones \ninstitucionales en el marco del proceso de transformación digital en la Municipalidad \nProvincial de Calca. \nCONCLUSIÓN \nPor lo expuesto, se solicita de manera formal la emisión del informe legal \ncorrespondiente, a fin de continuar con el procedimiento de aprobación del Plan de \nGobierno Digital de la Municipalidad Provincial de Calca. \nSin otro particular, quedo atento a cualquier consulta o documentación adicional que se \nrequiera para el trámite correspondiente. \nAtentamente", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 2000, repetition_penalty=1.2, use_cache = True)
tokenizer.batch_decode(outputs)

['<|begin_of_text|>Redacta un informe legal siguiendo estrictamente la estructura:\n\nI. ANTECEDENTES:\nII. BASE LEGAL:\nIII. ANÁLISIS:\nIV. CONCLUSIÓN:\nV. RECOMENDACIONES:\n\nAsegúrate de *no terminar el texto sin incluir todas las secciones*.\nFinaliza siempre con RECOMENDACIONES.\n\n### Instruction:\nAprobacion de plan de gobierno digital\n\n### Input:\nINFORME N°094-2025-UEI/OPP/MPC-BPR \nA        : Lic. Edder Joshimar Warthon Gamarra \nJefe de la Oficina de Planificación y Presupuesto \nDE      : Brady Palma Rodríguez \nEncargado de la Unidad de Estadística e Informática \nASUNTO : Solicitud de informe legal para aprobación del Plan de Gobierno Digital \nREFERENCIA : Plan de Gobierno Digital – Municipalidad Provincial de Calca, versión \npreliminar 2025 \nFECHA : Calca, 02 de agosto del 2025 \nMe dirijo a usted para saludarlo cordialmente y, a la vez, informar que la Unidad de \nEstadística e Informática ha culminado la elaboración del documento preliminar del Plan \nde Gobierno 

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
# Guardar el modelo y el tokenizer en carpeta local
output_dir = "/content/drive/MyDrive/informes_legales_ai2/modelo_finetuned2"

trainer.save_model(output_dir)        # Guarda pesos, config y tokenizer
tokenizer.save_pretrained(output_dir) # Guarda el tokenizer

('/content/drive/MyDrive/informes_legales_ai2/modelo_finetuned2/tokenizer_config.json',
 '/content/drive/MyDrive/informes_legales_ai2/modelo_finetuned2/special_tokens_map.json',
 '/content/drive/MyDrive/informes_legales_ai2/modelo_finetuned2/tokenizer.json')

In [ ]:
from unsloth import FastLanguageModel

# Cargar tu modelo entrenado
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = output_dir,
    max_seq_length = 8192,
    dtype = None,
    load_in_4bit = True,
)

# Exportar fusionado (ya con LoRA aplicado al modelo base)
merged_dir = "/content/drive/MyDrive/informes_legales_ai2/modelo_fusionado2"
model.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)

print("✅ Modelo fusionado guardado en:", merged_dir)

==((====))==  Unsloth 2026.8.3: Fast Llama patching. Transformers: 4.55.4.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Modelo fusionado guardado en: /content/drive/MyDrive/informes_legales_ai2/modelo_fusionado2
